# vLLM on an NVIDIA T4: benchmark on Google Colab

This notebook measures the summarization API on a **Tesla T4 (16 GB)**, the same GPU as an AWS `g4dn.xlarge`. The EC2 GPU deployment of this project is ready (`docker-compose.yml`) but waits for a GPU quota; these numbers stand in for it until then.

The request path is the same as in production: `scripts/benchmark.py` → FastAPI gateway → vLLM's OpenAI-compatible server. Two models are measured:

| Model | Why |
|-------|-----|
| `Qwen/Qwen2.5-1.5B-Instruct` | Same model as the EC2 CPU deployment: apples-to-apples comparison |
| `Qwen/Qwen3-4B-Instruct-2507` | Original target model of the GPU stack |

**Before you start**

- *Runtime* → *Change runtime type* → **T4 GPU**, then *Run all*.
- Free Colab GPUs are **not guaranteed**: a T4 may be unavailable at busy times, and sessions are disconnected after a period of inactivity or a maximum lifetime. If the runtime has no GPU, the first code cell stops with an explicit error.
- Expect the installation to take a few minutes, and each model a few more minutes to load before its benchmark starts.
- Results are written to `/content/results/` and offered as a download at the end.

## 1. Hardware

Records the GPU, its memory and the driver version. The driver decides which vLLM build can run (next step).

In [ ]:
import subprocess

try:
    query = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
except (FileNotFoundError, subprocess.CalledProcessError):
    raise SystemExit("No NVIDIA GPU found: select Runtime > Change runtime type > T4 GPU.")

GPU_NAME, GPU_MEMORY, DRIVER = [field.strip() for field in query.split(",")]
DRIVER_MAJOR = int(DRIVER.split(".")[0])
if "T4" not in GPU_NAME:
    raise SystemExit(f"Expected a T4, got {GPU_NAME}: results would not be comparable.")

HARDWARE = f"Google Colab, NVIDIA {GPU_NAME} ({GPU_MEMORY}), driver {DRIVER}"
print(HARDWARE)
!nvidia-smi

## 2. Configuration

Same vLLM version and flags as `docker-compose.yml`. The T4 (Turing, compute capability 7.5) has **no bfloat16 support**, and both checkpoints are stored in bf16, so vLLM casts the weights to fp16 (`--dtype=half`). Note that the EC2 CPU run uses bf16: same 16-bit size, different number format.

The GPU concurrency levels (1, 4, 16, 32) are the benchmark defaults; the CPU run used 1, 2, 4, 8.

In [ ]:
VLLM_VERSION = "0.31.0"
REPO_URL = "https://github.com/ALI-AL-MARJANI/Fast-LLM-summarization-api.git"
REPO_DIR = "/content/Fast-LLM-summarization-api"

MODELS = [
    "Qwen/Qwen2.5-1.5B-Instruct",   # same model as the EC2 CPU deployment
    "Qwen/Qwen3-4B-Instruct-2507",  # original target model of the GPU stack
]
VLLM_ARGS = ["--dtype=half", "--max-model-len=8192", "--gpu-memory-utilization=0.90"]

CONCURRENCY = [1, 4, 16, 32]
REQUESTS_PER_LEVEL = 64
MAX_TOKENS = 128

# Local ports. 8080 (used by the EC2 stacks) is taken by the Colab runtime itself.
VLLM_PORT = 8000
GATEWAY_PORT = 8090

## 3. Install vLLM in an isolated environment

**Colab conflict:** Colab ships its own PyTorch build, while vLLM pins an exact version (`torch==2.13.0` for vLLM 0.31.0) because its compiled CUDA kernels only work with that build. Installing vLLM into Colab's Python would replace torch and break preinstalled packages such as torchvision. vLLM's documentation recommends a fresh environment, so vLLM gets **its own virtual environment** (created with `uv`, which is much faster than pip), and Colab's environment is left untouched.

**CUDA build:** the default vLLM wheel on PyPI is built with CUDA 13, which needs an NVIDIA driver ≥ R580. On an older driver, the official `+cu129` wheel from the vLLM GitHub release is installed instead, with the matching CUDA 12.9 PyTorch build.

In [ ]:
VLLM_ENV = "/content/venv-vllm"

if DRIVER_MAJOR >= 580:
    VLLM_SPEC, TORCH_BACKEND = f"vllm=={VLLM_VERSION}", "cu130"
else:
    VLLM_SPEC = (
        f"https://github.com/vllm-project/vllm/releases/download/v{VLLM_VERSION}/"
        f"vllm-{VLLM_VERSION}+cu129-cp38-abi3-manylinux_2_28_x86_64.whl"
    )
    TORCH_BACKEND = "cu129"
print(f"Driver {DRIVER}: installing {VLLM_SPEC} with the {TORCH_BACKEND} PyTorch build")

!pip install -q uv
!uv venv -q --allow-existing --python 3.12 {VLLM_ENV}
!uv pip install --python {VLLM_ENV}/bin/python "{VLLM_SPEC}" --torch-backend={TORCH_BACKEND}

In [ ]:
# Record the versions actually installed, and check that this torch build sees the GPU.
ENGINE = subprocess.run(
    [f"{VLLM_ENV}/bin/python", "-c",
     "import torch, vllm; assert torch.cuda.is_available(), 'torch cannot see the GPU'; "
     "print(f'vLLM {vllm.__version__}, torch {torch.__version__} (CUDA {torch.version.cuda})')"],
    capture_output=True, text=True,
)
if ENGINE.returncode != 0:
    raise SystemExit(ENGINE.stderr)
ENGINE = ENGINE.stdout.strip()
print(ENGINE)

## 4. Get the project and install the gateway

The repository provides the gateway and `scripts/benchmark.py`. The gateway gets its own small environment too, with the pinned versions of `gateway/requirements.txt`.

In [ ]:
GATEWAY_ENV = "/content/venv-gateway"

!test -d {REPO_DIR} || git clone -q --depth 1 {REPO_URL} {REPO_DIR}
!uv venv -q --allow-existing --python 3.12 {GATEWAY_ENV}
!uv pip install -q --python {GATEWAY_ENV}/bin/python -r {REPO_DIR}/gateway/requirements.txt

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
print(f"Repository at commit {COMMIT}")

## 5. Helpers: start, wait, benchmark, stop

Both servers run as background processes, each in its own process group so that stopping it also stops its child processes (vLLM starts an engine process). Their output goes to `/content/logs/`.

For each model, `run_model`:

1. starts vLLM and waits until `/health` answers (weights download + loading + CUDA graph capture);
2. starts the gateway pointing to it, with the model name read by vLLM;
3. runs `scripts/benchmark.py` against the gateway and saves the Markdown report;
4. stops both servers and checks that the GPU memory is released, so the next model gets the whole GPU.

In [ ]:
import os
import secrets
import signal
import socket
import time
import urllib.request
from pathlib import Path

from IPython.display import Markdown, display

LOG_DIR = Path("/content/logs")
RESULTS_DIR = Path("/content/results")
LOG_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)
API_KEY = secrets.token_hex(32)


def start(name, cmd, port, env=None, cwd=None):
    with socket.socket() as sock:
        if sock.connect_ex(("127.0.0.1", port)) == 0:
            raise RuntimeError(f"Port {port} is already in use: change it in the configuration cell.")
    log = open(LOG_DIR / f"{name}.log", "w")
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, cwd=cwd,
                            env={**os.environ, **(env or {})}, start_new_session=True)


def tail(name, lines=30):
    return "\n".join((LOG_DIR / f"{name}.log").read_text(errors="replace").splitlines()[-lines:])


def wait_ready(name, proc, url, timeout_s):
    start_time = time.monotonic()
    while time.monotonic() - start_time < timeout_s:
        if proc.poll() is not None:
            raise RuntimeError(f"{name} exited with code {proc.returncode}:\n{tail(name)}")
        try:
            with urllib.request.urlopen(url, timeout=3):
                print(f"{name} ready after {time.monotonic() - start_time:.0f} s")
                return
        except OSError:
            time.sleep(5)  # not listening yet, or 503 while loading
    raise TimeoutError(f"{name} not ready after {timeout_s} s:\n{tail(name)}")


def stop(proc):
    if proc is None or proc.poll() is not None:
        return
    os.killpg(proc.pid, signal.SIGTERM)
    try:
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()


def log_lines(name, *needles):
    text = (LOG_DIR / f"{name}.log").read_text(errors="replace")
    return [line.strip() for line in text.splitlines()
            if any(needle in line.lower() for needle in needles)]


def run_model(model):
    slug = model.split("/")[-1].lower()
    vllm_name, gateway_name = f"vllm-{slug}", f"gateway-{slug}"
    vllm = gateway = None
    try:
        vllm = start(vllm_name, [f"{VLLM_ENV}/bin/vllm", "serve", model, *VLLM_ARGS,
                                 "--host=127.0.0.1", f"--port={VLLM_PORT}"], VLLM_PORT)
        wait_ready(vllm_name, vllm, f"http://127.0.0.1:{VLLM_PORT}/health", timeout_s=1200)

        gateway = start(
            gateway_name,
            [f"{GATEWAY_ENV}/bin/uvicorn", "app.main:create_app", "--factory",
             "--host=127.0.0.1", f"--port={GATEWAY_PORT}", "--no-access-log"],
            GATEWAY_PORT,
            env={"API_KEY": API_KEY, "VLLM_BASE_URL": f"http://127.0.0.1:{VLLM_PORT}",
                 "MODEL_NAME": model, "VLLM_REQUEST_TIMEOUT_S": "180", "LOG_LEVEL": "WARNING"},
            cwd=f"{REPO_DIR}/gateway",
        )
        wait_ready(gateway_name, gateway, f"http://127.0.0.1:{GATEWAY_PORT}/health", timeout_s=60)

        bench = subprocess.Popen(
            [f"{GATEWAY_ENV}/bin/python", f"{REPO_DIR}/scripts/benchmark.py",
             "--url", f"http://127.0.0.1:{GATEWAY_PORT}", "--hardware", HARDWARE,
             "--concurrency", *map(str, CONCURRENCY),
             "--requests", str(REQUESTS_PER_LEVEL), "--max-tokens", str(MAX_TOKENS)],
            env={**os.environ, "API_KEY": API_KEY},
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True,
        )
        for line in bench.stderr:  # progress, one line per concurrency level
            print(line, end="")
        table = bench.stdout.read()
        if bench.wait() != 0:
            raise RuntimeError(f"benchmark failed with code {bench.returncode}")

        report = (f"**Engine:** {ENGINE}, `{' '.join(VLLM_ARGS)}`, repository commit {COMMIT}  \n"
                  + table)
        path = RESULTS_DIR / f"colab-t4-{slug}.md"
        path.write_text(report)
        display(Markdown(report))
        print(f"Saved to {path}\n")
        print("\n".join(log_lines(vllm_name, "attention backend", "kv cache", "maximum concurrency")))
    finally:
        stop(gateway)
        stop(vllm)
        time.sleep(5)
        !nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## 6. Model 1: `Qwen/Qwen2.5-1.5B-Instruct`

Same model as the EC2 CPU deployment. The last lines show the attention backend vLLM selected (FlashAttention needs compute capability 8.0, so a T4 falls back to another backend) and the KV-cache size. After the servers stop, the GPU memory used should be back near 0 MiB.

In [ ]:
run_model(MODELS[0])

## 7. Model 2: `Qwen/Qwen3-4B-Instruct-2507`

The original target model: about 8 GB of fp16 weights, which leaves less memory for the KV-cache on a 16 GB T4.

In [ ]:
run_model(MODELS[1])

## 8. Collect the results

Prints both reports and downloads `results.zip` (reports and server logs). To publish them, copy the `.md` files to the repository's `benchmarks/` directory.

In [ ]:
for path in sorted(RESULTS_DIR.glob("*.md")):
    display(Markdown(f"### {path.name}\n\n" + path.read_text()))

!cd /content && zip -qr results.zip results logs
try:
    from google.colab import files
    files.download("/content/results.zip")
except ImportError:
    print("Not running on Colab: results are in /content/results.zip")

## Troubleshooting

- **"No NVIDIA GPU found"**: the runtime is CPU-only. Change the runtime type to T4 GPU; if none is available, try again later.
- **Installation error mentioning CUDA or the driver**: check the driver printed in step 1, and that step 3 picked the matching build.
- **"Port ... is already in use"**: another process listens on that port; pick another value for `VLLM_PORT` or `GATEWAY_PORT` in the configuration cell.
- **A server does not become ready**: the error shows the end of its log; the full logs are in `/content/logs/`.
- **Out of memory on the GPU**: make sure the previous model's servers stopped (step 5 prints the GPU memory used after each run), or restart the runtime and run all cells again.
- **Session disconnected**: the runtime is reset; run all cells again (the installation is not kept).